Exploratory Data Analysis for churn prediction of paid subscribers.


In [2]:
import pandas as pd
import numpy as np

In [80]:
df = pd.read_csv('lumen_subscriptions.csv')
df.describe()

,user_id,monthly_price_eur,tenure_months,monthly_watch_hours_90d,sessions_90d,unique_titles_90d,support_tickets_90d
count,3.000000e+04,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,2.015000e+06,11.922800,30.040467,16.690479,30.102733,10.069833,0.402967
std,8.660398e+03,2.803809,17.048978,12.257049,22.617411,7.876409,0.633086
min,2.000000e+06,9.990000,1.000000,0.590000,0.000000,0.000000,0.000000
25%,2.007500e+06,9.990000,15.000000,8.690000,16.000000,5.000000,0.000000
50%,2.015000e+06,9.990000,30.000000,13.510000,25.000000,9.000000,0.000000
75%,2.022499e+06,15.990000,45.000000,20.980000,38.000000,13.000000,1.000000
max,2.029999e+06,15.990000,59.000000,282.000000,508.000000,166.000000,4.000000


In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  30000 non-null  int64  
 1   plan                     30000 non-null  object 
 2   monthly_price_eur        30000 non-null  float64
 3   tenure_months            30000 non-null  int64  
 4   signup_channel           30000 non-null  object 
 5   country                  30000 non-null  object 
 6   payment_method           30000 non-null  object 
 7   monthly_watch_hours_90d  30000 non-null  float64
 8   sessions_90d             30000 non-null  int64  
 9   unique_titles_90d        30000 non-null  int64  
 10  support_tickets_90d      30000 non-null  int64  
 11  family_account           30000 non-null  bool   
 12  auto_renew_on            30000 non-null  bool   
 13  churned_30d              30000 non-null  bool   
dtypes: bool(3), float64(2)

In [98]:
base_churn = float(df['churned_30d'].mean()) # proportion of true values in col
print(f"Base churn rate: {base_churn:.2f}")

Base churn rate: 0.11


In [84]:
df['churned_30d'].value_counts(normalize=True) # returns relative frequency

,proportion
churned_30d,
False,0.8882
True,0.1118


In [85]:
# standardize numnerical features using z-scoring and group by 'churned_30d'
numeric_cols = ['monthly_price_eur', 'tenure_months', 'monthly_watch_hours_90d', 'sessions_90d', 'unique_titles_90d', 'support_tickets_90d']

z = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()

z['churned_30d'] = df['churned_30d']

conditional_means = z.groupby('churned_30d')[numeric_cols].mean()
conditional_means

,monthly_price_eur,tenure_months,monthly_watch_hours_90d,sessions_90d,unique_titles_90d,support_tickets_90d
churned_30d,,,,,,
False,0.001561,0.051427,0.019538,0.019518,0.018350,-0.042649
True,-0.012400,-0.408566,-0.155217,-0.155059,-0.145783,0.338824


In [99]:
# identify top features with largest mean gap
mean_diff = conditional_means.loc[True] - conditional_means.loc[False]
mean_diff.abs().sort_values(ascending=False).head(3)

,0
tenure_months,0.459993
support_tickets_90d,0.381473
monthly_watch_hours_90d,0.174754


In [87]:
# churn rate by category -- flag cats with 2x churn rate
categorical = ['plan', 'signup_channel', 'country', 'payment_method']

for col in categorical:
  results = df.groupby(col)['churned_30d'].agg(['mean', 'count'])
  results.sort_values('mean', ascending = True)
  print(results['mean'] >= (2 * base_churn)) # flag values

plan
premium     False
standard    False
Name: mean, dtype: bool
signup_channel
ads          False
app_store    False
organic      False
referral     False
Name: mean, dtype: bool
country
AU    False
BR    False
CA    False
DE    False
ES    False
FR    False
GB    False
IT    False
JP    False
NL    False
SE    False
US    False
Name: mean, dtype: bool
payment_method
apple_pay    False
card         False
gift_card    False
paypal       False
Name: mean, dtype: bool


In [88]:
# feature engineering
df['tickets_per_month'] = df['support_tickets_90d'] / 3
df['low_engagement'] = (df['monthly_watch_hours_90d'] < 2).astype(int)
df['new_user'] = (df['tenure_months'] <= 3)

In [89]:
# categorical encoding with pd.get_dummies() (use scikit-learn OneHotEncoder in production, for cats seen in training but not test)
df = pd.get_dummies(df, columns=['plan', 'signup_channel', 'payment_method'], drop_first=True) # drop_first to avoid redundant col

In [93]:
top_countries = df['country'].value_counts().head(8).index
df['country_grouped'] = df['country'].where(df['country'].isin(top_countries), other = 'OTHER')
df['country_grouped']

,country_grouped
0,OTHER
1,US
2,OTHER
3,OTHER
4,OTHER
...,...
29995,FR
29996,US
29997,FR
29998,US


In [96]:
# encode country_grouped & drop raw 'country' col
#df = pd.get_dummies(df, columns = ['country_grouped'], drop_first=True)
#df = df.drop(columns=['country'])
df

,user_id,monthly_price_eur,tenure_months,monthly_watch_hours_90d,sessions_90d,unique_titles_90d,support_tickets_90d,family_account,auto_renew_on,churned_30d,...,payment_method_gift_card,payment_method_paypal,country_grouped_BR,country_grouped_DE,country_grouped_ES,country_grouped_FR,country_grouped_GB,country_grouped_IT,country_grouped_OTHER,country_grouped_US
0,2000000,15.99,6,9.15,16,3,0,True,True,False,...,False,True,False,False,False,False,False,False,True,False
1,2000001,9.99,21,11.94,19,7,1,False,False,True,...,False,False,False,False,False,False,False,False,False,True
2,2000002,15.99,43,14.76,30,7,0,True,False,False,...,False,True,False,False,False,False,False,False,True,False
3,2000003,15.99,25,26.23,44,20,0,False,True,False,...,False,True,False,False,False,False,False,False,True,False
4,2000004,9.99,10,8.90,22,6,1,False,False,False,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,2029995,15.99,38,13.60,26,9,1,False,True,False,...,False,True,False,False,False,True,False,False,False,False
29996,2029996,9.99,16,23.18,40,12,0,False,True,False,...,True,False,False,False,False,False,False,False,False,True
29997,2029997,9.99,4,13.64,32,2,1,True,True,False,...,False,False,False,False,False,True,False,False,False,False
29998,2029998,15.99,41,15.37,31,8,1,True,True,False,...,False,True,False,False,False,False,False,False,False,True


In [97]:
import os
from sklearn.model_selection import train_test_split

# train/test split & save, stratified to ensure that 'churned_30' class frequencies are approximately preserved
y = df['churned_30d'] # labels
X = df.drop(columns = ['churned_30d', 'user_id']) # data, drop labels and user_id cols

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, churn_rate: {y_train.mean():.2f}")
print(f"Test: {len(X_test)}, churn_rate: {y_test.mean():2f}")

Train: 24000, churn_rate: 0.11
Test: 6000, churn_rate: 0.111833


In [101]:
os.makedirs("data", exist_ok=True)
X_train.to_parquet("data/X_train.parquet")
X_test.to_parquet("data/X_test.parquet")
y_train.to_frame().to_parquet("data/y_train.parquet")
y_test.to_frame().to_parquet("data/y_test.parquet")